# Reward model — 1.5B (the biggest lever)

Qwen2.5-**1.5B**-Instruct backbone + cleaned UltraFeedback, **LoRA** (base frozen → fits the T4; the
final checkpoint is merged to a full model so eval loads it normally). Beats the 0.5B's 0.726?
RM-only, ~5-6 h. Reports accuracy on cleaned held-out + old H4.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -2

## Config

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-1.5B-Instruct'
DATA_CLEAN='argilla/ultrafeedback-binarized-preferences-cleaned'
DATA_H4='HuggingFaceH4/ultrafeedback_binarized'
RM_SAMPLES=6000          # 1.5B is ~3x slower per step; 6k keeps it ~5-6 h on a T4
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print(f'model={MODEL} (LoRA) dtype={DTYPE} samples={RM_SAMPLES}')

## Train 1.5B reward model (LoRA; merged to full on final save)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE -o model.use_lora=true \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.eval_split='train[:2000]' \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=2000 -o data.max_length=512 \
  -o train.epochs=1 -o train.batch_size=8 -o train.grad_accum=2 -o train.bf16=$BF16 \
  -o train.lr=1.0e-4 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o train.save_every=300 -o output_dir=checkpoints/reward_model

## Evaluate — cleaned held-out (vs 0.5B's 0.726) + old H4

In [ ]:
import subprocess
def run(c):
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-500:]);
    if o.returncode: print(o.stderr[-1200:])
    return o.stdout
rc=run(f"python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA_CLEAN} --split 'train[:2000]' --max-samples 2000 --batch-size 8")
rh=run(f"python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA_H4} --split test_prefs --max-samples 2000 --batch-size 8")
md=(f'# 1.5B reward-model results\n\n- backbone: `{MODEL}` (LoRA)\n- train data: `{DATA_CLEAN}` ({RM_SAMPLES} pairs)\n\n'
    f'## Accuracy on CLEANED held-out (vs 0.5B = 0.726)\n```\n{rc}\n```\n\n'
    f'## Accuracy on old H4 test\n```\n{rh}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\nSaved -> RESULTS.md')

### Read
If 1.5B cleaned-accuracy > 0.726, the bigger backbone paid off (expected ~+3-6 pts).